In [49]:
import yfinance
import pandas

In [50]:
cmp = yfinance.Ticker( "ANET" )

In [71]:
import datetime

date = datetime.date(year=2025, month=8, day=19)

In [72]:
data = yfinance.download( "COALINDIA.NS", interval="1m", start=date, period="1d" )

C:\Users\Meowmaster\AppData\Local\Temp\ipykernel_1456\2324482203.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yfinance.download( "COALINDIA.NS", interval="1m", start=date, period="1d" )
[*********************100%***********************]  1 of 1 completed


In [62]:
data.columns = data.columns.get_level_values(0)

In [63]:
def gap_and_go_strategy(metrics, price, time, positions, funds, history):
    # Assume history is a pandas DataFrame with at least 'open', 'close', 'volume' columns
    # Only act in the first few minutes (e.g., first 15 minutes)

    # Parameters
    MIN_GAP = 2.0  # Minimum gap up percent to trade
    MAX_MINUTES = 15  # Only trade at open
    MAX_BUY = 0.2  # Max percent of cash to use per trade
    
    # 1. Calculate gap at market open
    if len(history) < 2:
        return {"action": "hold"}
    prev_close = history.iloc[-2]['close']
    curr_open = history.iloc[-1]['open']
    gap_percent = ((curr_open - prev_close) / prev_close) * 100

    # 2. Act only in first MAX_MINUTES after open
    market_open_time = history.iloc[0]['time']  # assuming this is a datetime object
    minutes_since_open = (time - market_open_time).total_seconds() / 60
    
    # 3. Gap condition and time window
    if gap_percent > MIN_GAP and minutes_since_open <= MAX_MINUTES:
        # Only buy if not already holding
        if positions.get("stock", 0) == 0:
            # Example: buy as much as 20% of your cash allows
            qty = int((funds * MAX_BUY) // price)
            if qty > 0:
                return {"action": "buy", "quantity": qty}
    # 4. Exit if holding and price starts to reverse (simple trailing stop or fixed target)
    if positions.get("stock", 0) > 0:
        # Exit if price drops below entry or target met (e.g., 2% gain)
        entry_price = positions.get("entry_price", price)
        if price < entry_price * 0.98 or price > entry_price * 1.02:
            return {"action": "sell", "quantity": positions["stock"]}

    return {"action": "hold"}


In [64]:
def dud_algorithm(metrics, price, time, positions, funds, history):
    # Track a static counter attribute on the function to alternate actions
    if not hasattr(dud_algorithm, "counter"):
        dud_algorithm.counter = 0

    dud_algorithm.counter += 1

    # Buy on odd calls if enough funds, sell on even calls if holding
    if dud_algorithm.counter % 2 == 0:
        # Buy action - buy as much as 50% funds allow
        qty = int((funds * 0.5) // price)
        if qty > 0:
            return {"action": "buy", "quantity": qty}
        else:
            return {"action": "hold"}
    else:
        # Sell action - sell all holdings if any
        stock_qty = positions.get("quantity", 0)
        if stock_qty > 0:
            return {"action": "sell", "quantity": stock_qty}
        else:
            return {"action": "hold"}


In [65]:
def simple_momentum_algorithm(metrics, price, time, positions, funds, history):
    # Need at least 2 bars to compare
    if len(history) < 2:
        return {"action": "hold"}

    latest = history[-1]
    previous = history[-2]

    stock_qty = positions.get("quantity", 0)

    # Buy if latest close > previous close and have funds for at least 1 share
    if latest.Close > previous.Close and funds >= price:
        qty = int(funds // price)
        if qty > 0:
            return {"action": "buy", "quantity": qty}

    # Sell if latest close < previous close and holding shares
    if latest.Close < previous.Close and stock_qty > 0:
        return {"action": "sell", "quantity": stock_qty}

    # Otherwise hold
    return {"action": "hold"}


In [66]:
def calculate_metrics(*args):
    pass



In [67]:
def algorithm(*args):
    return simple_momentum_algorithm( *args )

In [68]:
def resolve_action( action, holdings, price, funds ):
    if( action['action'] == "buy" ):
        quantity = action['quantity']
        amt = quantity*price
        funds -= amt
        holdings["average_price"] = ( ( holdings["average_price"] * holdings["quantity"] ) + amt ) /  ( quantity + holdings["quantity"] )
        holdings["quantity"] += quantity 

    elif( action['action'] == "sell" ):
        quantity = action['quantity']
        amt = quantity*price
        funds += amt
        holdings["quantity"] -= quantity 
        
    return funds

In [ ]:
from collections import namedtuple


def simulate( data, holdings, funds ):
    CandleStickTuple = namedtuple( "CandleStick", tuple( data.columns ),  )
    history = []
    for time, _series in data.iterrows():
        series = CandleStickTuple(*_series)
        metrics = calculate_metrics( history )
        action = algorithm( metrics, series.Close, time, holdings, funds, history )
        funds = resolve_action( action, holdings, series.Close, funds )
        history.append( series )

    return funds, holdings


In [86]:
from data_fetcher import fetch_data
import time
funds = 100_000

holdings = {
        "quantity" : 0,
        "average_price" : 0.0, 
}

before = funds  + holdings['average_price'] * holdings["quantity"]
days_earning = {}
for day in range(18,20):
    try:
        data = fetch_data( "BDL.NS", start=datetime.date( day=day,month=8,year=2025 ) )
        funds, holdings = simulate( data,holdings, funds )
    except:
        print(f" MAYBE CLOSE {day}")
    time.sleep(0.2)

    days_earning[day] = funds + holdings['average_price'] * holdings["quantity"]
after = funds + holdings['average_price'] * holdings["quantity"]

print( before )
print( after )

c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\data_fetcher.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yfinance.download( tick, interval = interval.value , start = start, period= '1d' )
[*********************100%***********************]  1 of 1 completed
c:\Users\Meowmaster\Desktop\PROject\IntradaySimulator\data_fetcher.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yfinance.download( tick, interval = interval.value , start = start, period= '1d' )
[*********************100%***********************]  1 of 1 completed


100000.0
98768.00305175781


In [87]:
print(days_earning)

{18: 101357.29382324219, 19: 98768.00305175781}
